
# DS4400 Training Models (Milestone)

This notebook covers the full pipeline for the College Scorecard earnings prediction project: loading the Field-of-Study dataset, handling privacy suppression tokens (`PS`), performing data cleaning and exploratory data analysis, training **Lasso** and **Random Forest** regressors, and evaluating performance with **MSE** and **R²**. All outputs are saved under `outputs/`.


In [35]:

import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

# Make Matplotlib caches writable in restricted environments
os.environ['MPLCONFIGDIR'] = os.path.join(os.getcwd(), '.mplconfig')
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

BASE_DIR = os.getcwd()
data_path = os.path.join(BASE_DIR, 'Most-Recent-Cohorts-Field-of-Study.csv')
out_dir = os.path.join(BASE_DIR, 'outputs')
fig_dir = os.path.join(out_dir, 'figures')
os.makedirs(fig_dir, exist_ok=True)

print('Data path:', data_path)
print('Figures dir:', fig_dir)


Data path: C:\Users\alexy\Desktop\jupyter notebook\DS4400\clone\Machine-Learning-Model\Most-Recent-Cohorts-Field-of-Study.csv
Figures dir: C:\Users\alexy\Desktop\jupyter notebook\DS4400\clone\Machine-Learning-Model\outputs\figures


In [54]:

# Load the full dataset (all columns read as strings to preserve 'PS' tokens)
df_raw = pd.read_csv(data_path, dtype=str, low_memory=False)
print(f'Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')


Raw dataset shape: 227,980 rows x 178 columns


## Data Cleaning

The following cleaning steps are applied before any EDA or modeling:
1. **Replace privacy-suppression tokens** (`PS`) with `NaN` across all columns.
2. **Convert numeric columns** to proper dtypes (coercing remaining non-numeric strings to `NaN`).
3. **Remove duplicate rows** (exact duplicates across all columns).
4. **Drop columns with >50% missingness** (as outlined in the proposal).
5. **Filter to rows where the target `EARN_MDN_4YR` is usable** (not null).
6. **Detect and flag outliers** in the target using the IQR method (retained for now).
7. **Report missingness summary** for the remaining features.

In [55]:

# Replace privacy-suppression tokens with NaN
PS_TOKEN = 'PS'
ps_counts = (df_raw == PS_TOKEN).sum()
total_ps = ps_counts.sum()
cols_with_ps = (ps_counts > 0).sum()
print(f'Total PS tokens replaced: {total_ps:,} across {cols_with_ps} columns')

df_clean = df_raw.replace(PS_TOKEN, np.nan)

# Convert columns that are predominantly numeric to proper dtypes
numeric_converted = []
for col in df_clean.columns:
    non_null = df_clean[col].dropna()
    if len(non_null) == 0:
        continue
    parsed = pd.to_numeric(non_null, errors='coerce')
    frac_numeric = parsed.notna().sum() / len(non_null)
    if frac_numeric > 0.5:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        numeric_converted.append(col)

print(f'Converted {len(numeric_converted)} columns to numeric dtype')
print(f'Dataset shape after PS replacement: {df_clean.shape}')

Total PS tokens replaced: 28,889,816 across 166 columns
Converted 142 columns to numeric dtype
Dataset shape after PS replacement: (227980, 178)


In [56]:

# Remove duplicate columns (if any exist in the raw CSV)
dup_cols = df_clean.columns[df_clean.columns.duplicated()]
if len(dup_cols) > 0:
    print(f'Duplicate column names found ({len(dup_cols)}): {list(dup_cols[:10])}...')
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]
    print(f'Shape after removing duplicate columns: {df_clean.shape}')
else:
    print('No duplicate column names found')

# Remove duplicate rows
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_after = len(df_clean)
print(f'Duplicate rows removed: {n_before - n_after:,}  (rows before: {n_before:,}, after: {n_after:,})')

# Filter to usable target rows
df_model = df_clean[df_clean['EARN_MDN_4YR'].notna()].copy()
print(f'Rows with usable EARN_MDN_4YR: {len(df_model):,} out of {len(df_clean):,}')

No duplicate column names found
Duplicate rows removed: 0  (rows before: 227,980, after: 227,980)
Rows with usable EARN_MDN_4YR: 58,102 out of 227,980


In [57]:

# Drop columns with >50% missingness
miss_frac = df_clean.isna().mean()
high_miss_cols = miss_frac[miss_frac > 0.5].index.tolist()
if 'EARN_MDN_4YR' in high_miss_cols:
    high_miss_cols.remove('EARN_MDN_4YR')

print(f'\nColumns with >50% missing: {len(high_miss_cols)} out of {len(df_clean.columns)}')
print('Examples:', high_miss_cols[:10])

df_clean = df_clean.drop(columns=high_miss_cols)
print(f'Shape after dropping high-missingness columns: {df_clean.shape}')

# Outlier detection on target (IQR method)
Q1 = df_model['EARN_MDN_4YR'].quantile(0.25)
Q3 = df_model['EARN_MDN_4YR'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outlier_mask = (df_model['EARN_MDN_4YR'] < lower_bound) | (df_model['EARN_MDN_4YR'] > upper_bound)
n_outliers = outlier_mask.sum()

print(f'\nTarget IQR outlier check:')
print(f'  Q1={Q1:,.0f}, Q3={Q3:,.0f}, IQR={IQR:,.0f}')
print(f'  Lower bound: {lower_bound:,.0f}, Upper bound: {upper_bound:,.0f}')
print(f'  Outliers flagged: {n_outliers:,} ({n_outliers/len(df_model)*100:.1f}%)')

# Missingness summary for remaining features
miss_summary = df_model.isna().mean().sort_values(ascending=False)
miss_nonzero = miss_summary[miss_summary > 0]
print(f'\nFeatures with any remaining missingness: {len(miss_nonzero)} out of {len(df_model.columns)}')
print(miss_nonzero.head(15).to_string())


Columns with >50% missing: 157 out of 178
Examples: ['DEBT_ALL_STGP_ANY_N', 'DEBT_ALL_STGP_ANY_MEAN', 'DEBT_ALL_STGP_ANY_MDN', 'DEBT_ALL_STGP_EVAL_N', 'DEBT_ALL_STGP_EVAL_MEAN', 'DEBT_ALL_STGP_EVAL_MDN', 'DEBT_ALL_PP_ANY_N', 'DEBT_ALL_PP_ANY_MEAN', 'DEBT_ALL_PP_ANY_MDN', 'DEBT_ALL_PP_EVAL_N']
Shape after dropping high-missingness columns: (227980, 21)

Target IQR outlier check:
  Q1=46,903, Q3=77,324, IQR=30,420
  Lower bound: 1,272, Upper bound: 122,954
  Outliers flagged: 2,277 (3.9%)

Features with any remaining missingness: 167 out of 178
DEBT_MALE_PP_EVAL_MDN        0.969571
DEBT_NOTMALE_PP_EVAL_MDN     0.969571
EARN_COUNT_HIGH_CRED_1YR     0.968693
DEBT_NOTMALE_PP_ANY_MDN      0.961241
DEBT_MALE_PP_ANY_MDN         0.961241
DEBT_PELL_PP_EVAL_MDN        0.951981
DEBT_NOPELL_PP_EVAL_MDN      0.951981
DEBT_MALE_PP_EVAL_MEAN       0.948298
DEBT_NOTMALE_PP_EVAL_MEAN    0.948229
DEBT_NOPELL_PP_ANY_MDN       0.939021
DEBT_PELL_PP_ANY_MDN         0.939021
DEBT_NOPELL_PP_EVAL_MEAN     0.9

In [58]:
# Check missingness of debt features in current df_model
debt_cols = [
    'DEBT_ALL_STGP_EVAL_MDN',
    'DEBT_ALL_STGP_EVAL_MEAN', 
    'DEBT_ALL_STGP_EVAL_MDN10YRPAY',
    'DEBT_ALL_STGP_ANY_MDN',
    'DEBT_ALL_STGP_ANY_MEAN',
]
print("Missingness of debt features in df_model:")
for col in debt_cols:
    if col in df_model.columns:
        rate = df_model[col].isna().mean()
        print(f"  {col}: {rate:.3f}")
    else:
        print(f"  {col}: NOT IN DATASET")

Missingness of debt features in df_model:
  DEBT_ALL_STGP_EVAL_MDN: 0.254
  DEBT_ALL_STGP_EVAL_MEAN: 0.333
  DEBT_ALL_STGP_EVAL_MDN10YRPAY: 0.254
  DEBT_ALL_STGP_ANY_MDN: 0.375
  DEBT_ALL_STGP_ANY_MEAN: 0.495


In [59]:

# Summary of cleaned dataset
print('Cleaned Dataset Summary')
print(f'Rows:    {len(df_model):,}')
print(f'Columns: {len(df_model.columns)}')
print(f'\nColumn dtypes:')
print(df_model.dtypes.value_counts().to_string())

print(f'\nTarget (EARN_MDN_4YR) statistics:')
print(df_model['EARN_MDN_4YR'].describe().to_string())

# List remaining columns grouped by type
num_cols_remaining = df_model.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_remaining = df_model.select_dtypes(exclude=[np.number]).columns.tolist()
print(f'\nNumeric columns ({len(num_cols_remaining)}): {num_cols_remaining[:20]}{"..." if len(num_cols_remaining)>20 else ""}')
print(f'Categorical/string columns ({len(cat_cols_remaining)}): {cat_cols_remaining[:20]}{"..." if len(cat_cols_remaining)>20 else ""}')

Cleaned Dataset Summary
Rows:    58,102
Columns: 178

Column dtypes:
float64    137
object      36
int64        5

Target (EARN_MDN_4YR) statistics:
count     58102.000000
mean      64909.914048
std       27730.067669
min        6917.000000
25%       46903.250000
50%       59017.500000
75%       77323.750000
max      336392.000000

Numeric columns (142): ['UNITID', 'OPEID6', 'MAIN', 'CIPCODE', 'CREDLEV', 'IPEDSCOUNT1', 'IPEDSCOUNT2', 'DEBT_ALL_STGP_ANY_N', 'DEBT_ALL_STGP_ANY_MEAN', 'DEBT_ALL_STGP_ANY_MDN', 'DEBT_ALL_STGP_EVAL_N', 'DEBT_ALL_STGP_EVAL_MEAN', 'DEBT_ALL_STGP_EVAL_MDN', 'DEBT_ALL_PP_ANY_N', 'DEBT_ALL_PP_ANY_MEAN', 'DEBT_ALL_PP_ANY_MDN', 'DEBT_ALL_PP_EVAL_N', 'DEBT_ALL_PP_EVAL_MEAN', 'DEBT_ALL_PP_EVAL_MDN', 'DEBT_MALE_STGP_ANY_N']...
Categorical/string columns (36): ['INSTNM', 'CONTROL', 'CIPDESC', 'CREDDESC', 'BBRR2_FED_COMP_DFLT', 'BBRR2_FED_COMP_DLNQ', 'BBRR2_FED_COMP_FBR', 'BBRR2_FED_COMP_DFR', 'BBRR2_FED_COMP_NOPROG', 'BBRR2_FED_COMP_MAKEPROG', 'BBRR2_FED_COMP_PAIDINFUL

## Exploratory Data Analysis (EDA)

With cleaned data in place, the following figures explore the target distribution and its relationships with key predictors.

In [68]:

# EDA figures
sns.set_theme(style='whitegrid')

y = df_model['EARN_MDN_4YR'].values

# Figure 1: target distribution (histogram + KDE)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(y, bins=40, kde=True, ax=ax)
ax.set_title('Distribution of EARN_MDN_4YR')
ax.set_xlabel('Median Earnings (4yr)')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, 'target_hist.png'), dpi=200)
plt.close(fig)

# Figure 2: mean earnings by CONTROL (top groups by count)
if 'CONTROL' in df_model.columns:
    mean_by_control = df_model.groupby('CONTROL', dropna=False)['EARN_MDN_4YR'].mean().sort_values(ascending=False)
    count_by_control = df_model['CONTROL'].value_counts(dropna=False)
    top_controls = list(count_by_control.index[:5])
    subset_controls = mean_by_control.loc[[c for c in top_controls if c in mean_by_control.index]]

    plt.figure(figsize=(8, 4.5))
    sns.barplot(x=subset_controls.index.astype(str), y=subset_controls.values)
    plt.xticks(rotation=25, ha='right')
    plt.title('Mean EARN_MDN_4YR by Institution Type (CONTROL)')
    plt.xlabel('CONTROL')
    plt.ylabel('Mean EARN_MDN_4YR')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'mean_by_control.png'), dpi=200)
    plt.close()

# Figure 3: debt vs earnings (only if column survived cleaning)
debt_col = 'DEBT_ALL_STGP_ANY_MDN'
if debt_col in df_model.columns:
    subset_debt = df_model[df_model[debt_col].notna()]
    plt.figure(figsize=(6.8, 4.8))
    sns.scatterplot(x=debt_col, y='EARN_MDN_4YR', data=subset_debt, s=10, alpha=0.4)
    plt.title(f'{debt_col} vs EARN_MDN_4YR (n={len(subset_debt):,})')
    plt.xlabel(debt_col)
    plt.ylabel('EARN_MDN_4YR')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'debt_vs_earnings.png'), dpi=200)
    plt.close()
else:
    print(f'{debt_col} was dropped during cleaning (>50% missing)')

# Figure 4: correlation heatmap among numeric features
selected_for_corr = numeric_features + ['EARN_MDN_4YR']
corr_matrix = df_model[selected_for_corr].corr()

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=ax, square=False, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (selected features vs EARN_MDN_4YR)', 
             fontsize=13, pad=10)
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, 'correlation_heatmap.png'), dpi=200, bbox_inches='tight')
plt.close(fig)

# Figure 5: top CIPDESC by frequency within usable rows
if 'CIPDESC' in df_model.columns:
    cip_counts = df_model['CIPDESC'].fillna('Unknown').value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.barplot(x=cip_counts.values, y=cip_counts.index, ax=ax)
    ax.set_title('Top 10 Fields of Study by Frequency (usable target rows)', fontsize=13, pad=12)
    ax.set_xlabel('Count')
    ax.set_ylabel('CIPDESC')
    fig.subplots_adjust(left=0.45, top=0.90, bottom=0.12)
    fig.savefig(os.path.join(fig_dir, 'top_cipdesc.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)

# Figure 6: missingness rate bar chart for remaining features
selected_cols = numeric_features + ['EARN_MDN_4YR']
miss_selected = df_model[selected_cols].isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=miss_selected.values * 100, y=miss_selected.index, ax=ax)
ax.set_title('Missingness Rate of Selected Features (after target filtering)')
ax.set_xlabel('Missing %')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'missingness_bar.png'), dpi=200)
plt.close()

print('EDA figures saved to:', fig_dir)


EDA figures saved to: C:\Users\alexy\Desktop\jupyter notebook\DS4400\clone\Machine-Learning-Model\outputs\figures


## Feature Selection and Modeling

After cleaning, features are selected for modeling:
- **Numeric features**: columns that survived the >50% missingness threshold (excluding identifiers and categorical codes). Missing values are imputed with the median, then z-score scaled.
- **Categorical features**: `CONTROL`, `CREDLEV`, `DISTANCE`, `CIPCODE` — imputed with mode, then one-hot encoded.

**Normalization:** All continuous numeric features are standardized (z-score: zero mean, unit variance) before training. The target variable (`EARN_MDN_4YR`) is also standardized so that MSE is reported on a normalized scale, making model comparison more interpretable. After evaluation, predictions are inverse-transformed back to the original dollar scale for residual plots and RMSE reporting.

Two regression models are trained and evaluated on an 80/20 train/test split: **Lasso** and **Random Forest**.

In [65]:
# Modeling pipeline (all models)

# Columns treated as categorical (integer codes, not continuous)
candidate_categorical = ['CONTROL', 'CREDLEV', 'DISTANCE', 'CIPCODE']
categorical_features = [c for c in candidate_categorical if c in df_model.columns]

# Manually specified feature sets based on domain knowledge and correlation analysis
# Categorical features (treated as categories, not continuous values)
categorical_features = ['CONTROL', 'CREDLEV', 'DISTANCE', 'CIPCODE']

# Numeric features: selected based on relevance, missingness, and leakage checks
numeric_features = [
    # Program size / scale
    'IPEDSCOUNT1',                   # credentials conferred (year 1)
    'IPEDSCOUNT2',                   # credentials conferred (year 2)
    # Debt burden (core research variables)
    'DEBT_ALL_STGP_EVAL_MDN',        # median loan debt at graduating institution
    'DEBT_ALL_STGP_EVAL_MEAN',       # mean loan debt at graduating institution
    'DEBT_ALL_STGP_EVAL_MDN10YRPAY', # estimated monthly payment (10-yr plan)
    'DEBT_ALL_STGP_ANY_MDN',         # median cumulative debt (all institutions attended)
    # Early earnings signals (pre-4yr window, not leakage)
    'EARN_COUNT_NWNE_HI_1YR',        # count working and not enrolled at 1yr
    'EARN_COUNT_NWNE_1YR',           # count not working/not enrolled at 1yr
    # Repayment behavior
    'BBRR2_FED_COMP_N',              # number of borrowers in 2-yr repayment cohort
    'BBRR3_FED_COMP_N',              # number of borrowers in 3-yr repayment cohort
]

all_features = numeric_features + categorical_features
print(f'Numeric features selected ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features selected ({len(categorical_features)}): {categorical_features}')

X = df_model[all_features].copy()
y_raw = df_model['EARN_MDN_4YR'].values

# Normalize the target variable (z-score)
y_scaler = StandardScaler()
y = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

print(f'\nTarget normalization applied (StandardScaler):')
print(f'  Original  — mean: {y_raw.mean():,.1f}, std: {y_raw.std():,.1f}')
print(f'  Normalized — mean: {y.mean():.4f}, std: {y.std():.4f}')

# Feature preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'lasso': Lasso(alpha=0.001, max_iter=10000, random_state=42),
    'random_forest': RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),
    'ridge': Ridge(alpha=1.0),
    'gradient_boosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    )
}

results = {}
predictions_norm = {}
predictions_dollar = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred_norm = pipe.predict(X_test)

    mse_norm = mean_squared_error(y_test, pred_norm)
    r2 = r2_score(y_test, pred_norm)

    # Inverse-transform to dollar scale for interpretable RMSE
    pred_dollar = y_scaler.inverse_transform(pred_norm.reshape(-1, 1)).ravel()
    y_test_dollar = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
    rmse_dollar = np.sqrt(mean_squared_error(y_test_dollar, pred_dollar))

    results[name] = {
        'mse_normalized': float(mse_norm),
        'r2': float(r2),
        'rmse_dollar': float(rmse_dollar)
    }
    predictions_norm[name] = pred_norm
    predictions_dollar[name] = pred_dollar

    print(f'\n{name}:')
    print(f'  Normalized MSE: {mse_norm:.4f}')
    print(f'  R²:             {r2:.4f}')
    print(f'  RMSE (dollars): ${rmse_dollar:,.0f}')

Numeric features selected (10): ['IPEDSCOUNT1', 'IPEDSCOUNT2', 'DEBT_ALL_STGP_EVAL_MDN', 'DEBT_ALL_STGP_EVAL_MEAN', 'DEBT_ALL_STGP_EVAL_MDN10YRPAY', 'DEBT_ALL_STGP_ANY_MDN', 'EARN_COUNT_NWNE_HI_1YR', 'EARN_COUNT_NWNE_1YR', 'BBRR2_FED_COMP_N', 'BBRR3_FED_COMP_N']
Categorical features selected (4): ['CONTROL', 'CREDLEV', 'DISTANCE', 'CIPCODE']

Target normalization applied (StandardScaler):
  Original  — mean: 64,909.9, std: 27,729.8
  Normalized — mean: 0.0000, std: 1.0000

lasso:
  Normalized MSE: 0.3839
  R²:             0.6161
  RMSE (dollars): $17,181

random_forest:
  Normalized MSE: 0.2644
  R²:             0.7356
  RMSE (dollars): $14,259

ridge:
  Normalized MSE: 0.3152
  R²:             0.6848
  RMSE (dollars): $15,569

gradient_boosting:
  Normalized MSE: 0.3621
  R²:             0.6379
  RMSE (dollars): $16,687


In [69]:
# Residual plots (dollar scale for interpretability)
y_test_dollar = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
model_names = list(results.keys())
fig, axes = plt.subplots(1, len(model_names), figsize=(14, 5), sharey=True)

for ax, name in zip(axes, model_names):
    pred_d = predictions_dollar[name]
    residuals = y_test_dollar - pred_d
    ax.scatter(pred_d, residuals, s=10, alpha=0.35, edgecolors='none')
    ax.axhline(0, color='red', linewidth=1)
    ax.set_title(f'{name}  (R²={results[name]["r2"]:.4f})')
    ax.set_xlabel('Predicted Earnings ($)')

axes[0].set_ylabel('Residual ($)')
fig.suptitle('Residual Plots (test set, dollar scale)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'residuals_combined.png'), dpi=200, bbox_inches='tight')
plt.close()

# Random Forest feature importances (top features)
rf_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', models['random_forest'])])
rf_pipe.fit(X_train, y_train)
rf_model = rf_pipe.named_steps['model']
feature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()

importances = rf_model.feature_importances_
idx = np.argsort(importances)[-15:][::-1]

plt.figure(figsize=(9, 5))
sns.barplot(x=importances[idx], y=feature_names[idx])
plt.title('Top Random Forest Feature Importances (one-hot expanded)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'rf_feature_importances_top.png'), dpi=200)
plt.close()

# Gradient Boosting feature importances
gb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', models['gradient_boosting'])])
gb_pipe.fit(X_train, y_train)
gb_model = gb_pipe.named_steps['model']
feature_names_gb = gb_pipe.named_steps['preprocessor'].get_feature_names_out()

importances_gb = gb_model.feature_importances_
idx_gb = np.argsort(importances_gb)[-15:][::-1]

plt.figure(figsize=(9, 5))
sns.barplot(x=importances_gb[idx_gb], y=feature_names_gb[idx_gb])
plt.title('Top Gradient Boosting Feature Importances (one-hot expanded)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'gb_feature_importances_top.png'), dpi=200)
plt.close()

# Save metrics
payload = {
    'dataset': {
        'n_total_rows': int(len(df_raw)),
        'n_target_usable': int(len(df_model)),
        'target_mean': float(y_raw.mean()),
        'target_median': float(np.median(y_raw)),
        'target_std': float(y_raw.std(ddof=1)),
        'target_min': float(y_raw.min()),
        'target_max': float(y_raw.max()),
    },
    'normalization': {
        'method': 'StandardScaler (z-score)',
        'target_mean_used': float(y_scaler.mean_[0]),
        'target_std_used': float(y_scaler.scale_[0]),
    },
    'model_results': results,
    'model_test_split': {'test_size': 0.2, 'random_state': 42},
    'feature_sets': {
        'numeric': numeric_features,
        'categorical': categorical_features
    }
}

with open(os.path.join(out_dir, 'metrics.json'), 'w') as f:
    json.dump(payload, f, indent=2)

print('\nSaved metrics to:', os.path.join(out_dir, 'metrics.json'))



Saved metrics to: C:\Users\alexy\Desktop\jupyter notebook\DS4400\clone\Machine-Learning-Model\outputs\metrics.json
